# TissueMC: GPU rung R0 on a Colab T4

Builds the repository with CUDA, validates the GPU engine, compares it with the CPU on identical photons, runs the float32-totals demonstration, benchmarks, and prints **one summary block** at the end.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*

Then *Runtime → Run all*. It takes roughly 10 to 15 minutes. Nothing here needs passwords or keys. At the end, copy the block under `===== R0 SUMMARY =====` and paste it back.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
!git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git
%cd tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
# Configure and build with CUDA for the T4 (sm_75). A pip CMake is used because it knows the current CUDA.
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -5
!./build/tissuemc --version

In [ ]:
# The CPU test suite on this machine (the shared physics must still pass here).
!./build/tissuemc_tests 2>&1 | tail -4

In [ ]:
# 1. Validate the GPU engine against the same published values and oracle as the CPU engine.
!tissuemc-validate --binary build/tissuemc --engine cuda --report results/gpu-validation.json 2>&1 | grep -v '^  note' | tail -45

In [ ]:
# 2. Same photons through the CPU (float32) and the GPU: they should agree far more closely than noise.
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --photons 2000000 --report results/gpu-cross-engine.json

In [ ]:
# 3. Demonstration, not a validated mode: float32 running totals. Expect conservation to FAIL here.
!tissuemc-validate --binary build/tissuemc --engine cuda --gpu-tally float32 --cases v3_three_layer --report results/gpu-float32-tally.json 2>&1 | grep -E "conservation|rd |tt |checks" || true

In [ ]:
# 4. Benchmark (validates again first, then times the tissue-like workload).
!tissuemc-bench --engine cuda --binary build/tissuemc --output results/gpu-r0.json 2>&1 | tail -14

In [ ]:
import json
import pathlib
import subprocess


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


val, cross, f32, bench = (
    load(n)
    for n in (
        "gpu-validation.json",
        "gpu-cross-engine.json",
        "gpu-float32-tally.json",
        "gpu-r0.json",
    )
)
commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
summary = {
    "commit": commit,
    "device": bench["device"] if bench else None,
    "hardware": bench["hardware"] if bench else None,
    "validation": None
    if not val
    else {
        "checks": len(val),
        "passed": sum(r["passed"] for r in val),
        "failed": [
            f"{r['case']} {r['quantity']} z={r['z']:+.2f} ({r['source']})"
            for r in val
            if not r["passed"]
        ],
    },
    "cross_engine": None
    if not cross
    else {
        "agree": cross["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in cross["gaps"]},
    },
    "float32_totals_demo": None
    if not f32
    else [
        {"quantity": r["quantity"], "z": round(r["z"], 2), "passed": r["passed"]}
        for r in f32
        if r["quantity"] in ("conservation", "rd", "tt")
    ],
    "benchmark": None
    if not bench
    else {
        "photons": bench["photons"],
        "batches": bench["batches"],
        "validated_first": bench["validated_before_measuring"],
        "kernel_seconds": bench["kernel_seconds"],
        "median_kernel_s": bench["kernel"]["median"],
        "million_events_per_s": round(bench["events_per_second"] / 1e6, 1),
        "million_photons_per_s": round(bench["photons_per_second"] / 1e6, 3),
        "cpu_reference_million_events_per_s": {
            k: round(v / 1e6, 1)
            for k, v in (bench["cpu_reference"] or {"events_per_second_by_threads": {}})[
                "events_per_second_by_threads"
            ].items()
        },
    },
}
print("===== R0 SUMMARY =====")
print(json.dumps(summary, indent=1))